# Banking Intent Classification với Unsloth trên Google Colab

Fine-tunes **Llama-3.1 8B** cho task phân loại ý định ngân hàng (BANKING77) sử dụng **Unsloth + QLoRA**.

**Trước khi chạy:**
1. Vào **Runtime → Change runtime type → T4 GPU**
2. Chọn **Runtime → Run all**
3. Phê duyệt quyền truy cập Google Drive khi được hỏi

Nếu đã có checkpoint và sample_data trên Drive, notebook sẽ tự động bỏ qua bước tải dữ liệu và training.

## Step 0: Kết nối Google Drive

Mount Drive để lưu checkpoint bền vững. Thư mục `MyDrive/banking-intent-unsloth/` sẽ được tạo tự động.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/banking-intent-unsloth"
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results",     exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/sample_data", exist_ok=True)
print(f"Drive directory: {DRIVE_DIR}")

## Step 1: Cài đặt Dependencies

Cài Unsloth với xformers được pin theo đúng phiên bản PyTorch đang chạy trên Colab.

In [ ]:
%%capture
import torch, re

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

# datasets==2.21.0 is required — banking77 uses a loading script (banking77.py)
# which is no longer supported in datasets>=4.0
!pip install sentencepiece protobuf "datasets==2.21.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install scikit-learn pandas pyyaml matplotlib seaborn tqdm -q

In [ ]:
# Verify critical imports
from unsloth import FastModel
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import bitsandbytes
print("FastModel (Unsloth): OK")
print("peft:                OK")
print("bitsandbytes:        OK")

## Step 2: Clone Repository & Chuẩn bị môi trường

In [ ]:
import os

REPO_URL  = "https://github.com/tunah72/banking-intent-unsloth.git"
REPO_PATH = "/content/banking-intent-unsloth"

if os.path.exists(REPO_PATH):
    print("Repo already cloned. Pulling latest changes...")
    os.system(f"git -C {REPO_PATH} pull")
else:
    os.system(f"git clone {REPO_URL} {REPO_PATH}")

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")

## Step 3: Chuẩn bị dữ liệu

Nếu `sample_data/` đã có trên Drive → restore về local (nhanh hơn).
Nếu chưa có → tải BANKING77 từ HuggingFace và áp dụng Stratified Sampling:
- **Train:** 50 mẫu/nhãn (~3,826 tổng)
- **Validation:** 5 mẫu/nhãn (~375 tổng)
- **Test:** 10 mẫu/nhãn (770 tổng)

In [ ]:
import os, shutil

DRIVE_SAMPLE = f"{DRIVE_DIR}/sample_data"

if os.path.exists(DRIVE_SAMPLE) and os.listdir(DRIVE_SAMPLE):
    shutil.copytree(DRIVE_SAMPLE, "sample_data", dirs_exist_ok=True)
    print("Restored sample_data from Drive:", os.listdir("sample_data"))
else:
    print("sample_data not found on Drive. Downloading from HuggingFace...")
    os.system("python scripts/preprocess_data.py")
    # Sync to Drive for future use
    shutil.copytree("sample_data", DRIVE_SAMPLE, dirs_exist_ok=True)
    print(f"sample_data synced to Drive: {DRIVE_SAMPLE}")

## Step 4: Fine-tuning Mô hình

Fine-tunes `unsloth/Llama-3.1-8B-bnb-4bit` với **Unsloth FastModel + QLoRA** (standard PEFT):
- LoRA rank: 16, alpha: 32, dropout: 0
- modules_to_save: `["score"]` — classification head trainable
- Gradient checkpointing quản lý bởi Trainer (enable train / disable eval)
- Effective batch size: 8 (batch_size=2 × gradient_accumulation=4)
- Learning rate: 2e-4, warmup 5%, scheduler: linear
- Max sequence length: 128

**Nếu đã có checkpoint trên Drive → bỏ qua bước này.**

In [ ]:
import os, shutil

LOCAL_CKPT = "checkpoints"
DRIVE_CKPT = f"{DRIVE_DIR}/checkpoints"
FINAL_MODEL = os.path.join(LOCAL_CKPT, "final_best_model")

# Restore checkpoint from Drive if available
if os.path.exists(DRIVE_CKPT) and os.listdir(DRIVE_CKPT):
    shutil.copytree(DRIVE_CKPT, LOCAL_CKPT, dirs_exist_ok=True)
    print(f"Restored checkpoints from Drive → {LOCAL_CKPT}")
    print("Contents:", os.listdir(LOCAL_CKPT))
else:
    print("No checkpoints on Drive. Will train from scratch.")

In [ ]:
import os

FINAL_MODEL = "checkpoints/final_best_model"

if os.path.exists(FINAL_MODEL):
    print(f"Checkpoint found at '{FINAL_MODEL}'. Skipping training.")
else:
    print("No final checkpoint found. Starting training...")
    os.system("bash train.sh")
    # Save to Drive after training completes
    import shutil
    shutil.copytree("checkpoints", f"{DRIVE_DIR}/checkpoints", dirs_exist_ok=True)
    print(f"Checkpoint saved to Drive: {DRIVE_DIR}/checkpoints")

## Step 5: Inference Demo

Tải model đã fine-tune và dự đoán nhãn ý định cho một số câu banking mẫu.

In [ ]:
!bash inference.sh

## Step 6: Đánh giá Mô hình (Evaluation)

Đánh giá trên tập Test (770 mẫu) và tạo ra:
- `results/test_predictions.csv` — Log dự đoán đầy đủ
- `results/classification_report.txt` — Precision, Recall, F1 theo từng nhãn
- `results/metrics.json` — Metrics dạng JSON
- `results/confusion_matrix.png` — Confusion matrix heatmap

In [ ]:
!bash evaluate.sh

### Kết quả Đánh giá

In [ ]:
import json

with open("results/metrics.json", "r") as f:
    metrics = json.load(f)

macro    = metrics.get("macro avg", {})
accuracy = metrics.get("accuracy", 0)

print("=" * 50)
print(" FINAL EVALUATION METRICS")
print("=" * 50)
print(f"  Overall Accuracy : {accuracy * 100:.2f}%")
print(f"  Macro Precision  : {macro.get('precision', 0) * 100:.2f}%")
print(f"  Macro Recall     : {macro.get('recall',    0) * 100:.2f}%")
print(f"  Macro F1-Score   : {macro.get('f1-score',  0) * 100:.2f}%")
print("=" * 50)

In [ ]:
with open("results/classification_report.txt", "r") as f:
    print(f.read())

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/confusion_matrix.png", width=900))

## Step 7: Lưu Kết quả lên Google Drive

Sao chép toàn bộ output lên Drive để lưu trữ bền vững.
Sau khi chạy cell này, tất cả file sẽ nằm tại `MyDrive/banking-intent-unsloth/`.

In [ ]:
import shutil, os

def sync_to_drive(local_path, drive_subdir):
    dest = f"{DRIVE_DIR}/{drive_subdir}"
    if os.path.exists(local_path):
        shutil.copytree(local_path, dest, dirs_exist_ok=True)
        print(f"Saved {local_path} → {dest}")
    else:
        print(f"[SKIP] {local_path} not found")

sync_to_drive("results",     "results")
sync_to_drive("sample_data", "sample_data")

print(f"\nAll outputs saved to: {DRIVE_DIR}")